In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import SpringRank
import os

In [3]:
def filter_hiring(df, multigraph=False):
    """
    exclude faculty employed at level `X` who did not receive their degree from
    one of U.S. institutions that employs faculty at level `X`

    We require institutions to have at least one hire and at least one placement.

    20211213:
        Require institutions to employ at least one graduate in the field.
    
    If multigraph is false, will collapse the PersonIds into `HiredFaculty`. 
    Else leaves a row for each PersonId.
    """
    df = df.copy()

    df['HiredFaculty'] = df.groupby(['DegreeInstitutionID', 'InstitutionId'])['PersonId'].transform('nunique')
    df['TotalHires'] = df.groupby('InstitutionId')['PersonId'].transform('nunique')
    df['TotalPlacements'] = df.groupby('DegreeInstitutionID')['PersonId'].transform('nunique')

    df = df[
        (df['DegreeInstitutionID'].notnull())
        &
        (df['InstitutionId'].notnull())
        &
        (df['TotalHires'] > 0)
        &
        (df['TotalPlacements'] > 0)
        &
        (df['HiredFaculty'] > 0)
    ]

    while set(df['DegreeInstitutionID'].unique()) - set(df['InstitutionId'].unique()):
        df = df[
            df['DegreeInstitutionID'].isin(df['InstitutionId'].unique())
        ]

    if multigraph:
        df = df[
            [
                'DegreeInstitutionID',
                'InstitutionId',
                'PersonId',
            ]
        ].drop_duplicates()
    else:
        df = df[
            [
                'DegreeInstitutionID',
                'InstitutionId',
                'HiredFaculty',
            ]
        ].drop_duplicates()

    return df

def recover_ranks(df, recovered_rank_column='Rank'):
    """
    A[i, j] = x --> "institution j hired x faculty from institution i"

    1. give each InstitutionId a zero-index.
    2. map the DataFrame onto a matrix
    3. pass the matrix to SpringRank
    4. shift the ranks:
        - min(ranks) = 0
    5. map the ranks back from their indexes

    return the ranks
    """
    df = df.copy()

    identifiers = identifier_to_integer_id_map(df['InstitutionId'])

    df['i'] = df['DegreeInstitutionID'].apply(identifiers.get).astype(int)
    df['j'] = df['InstitutionId'].apply(identifiers.get).astype(int)
    matrix = np.zeros((len(identifiers), len(identifiers)), dtype=int)

    for i, j, w in zip(df['i'], df['j'], df['HiredFaculty']):
        matrix[i, j] = w

    index_sorted_identifiers = sorted(identifiers.keys(), key=lambda _id: identifiers[_id])
    ranks = pd.DataFrame({
        recovered_rank_column: SpringRank.get_scaled_ranks(matrix),
        'InstitutionId': index_sorted_identifiers,
    })

    min_rank = min(ranks[recovered_rank_column])
    ranks[recovered_rank_column] -= min_rank
    return ranks

def identifier_to_integer_id_map(identifiers):
    """
    maps each element in a list-like object to a unique 0-indexed integer. 

    in: a list like object
    out: dict. 
        - keys: each unique value in `identifiers`
        - values: a unique 0-indexed integer.  
    """
    identifiers = sorted(list(set(identifiers)))
    return {identifier: i for i, identifier in enumerate(identifiers)}

In [ ]:
# prepare the data

# read data
df = pd.read_csv(r"my_path\phd work major comparison\4. predict discipline for all\phd_prof_area.csv")

# concat institution
df_inst = pd.read_csv(r"my_path\university match\faculty_inst_phdinst.csv")
df_inst = df_inst.sort_values(['survey_year']).drop_duplicates('PersonId', keep='first')

df = pd.merge(df, df_inst, on='PersonId', how='inner').drop_duplicates()
df

,PersonId,id,phd_area,prof_area,InstitutionId,DegreeYear,DegreeInstitutionID,survey_year
0,1051864,1,Education,Education,348,2011,168,2011
1,181305,1,Education,Health,125,2011,538136,2011
2,349848,8,Computational Sciences,Engineering,12,2011,12,2011
3,378410,1,Education,Health,192,2011,192,2011
4,390738,17,Agriculture,Agriculture,270,2011,270,2011
...,...,...,...,...,...,...,...,...
60517,803513,20,"Architecture, Design, Planning","Architecture, Design, Planning",286,2016,45,2016
60518,843695,20,"Architecture, Design, Planning","Architecture, Design, Planning",301,2016,301,2017
60519,85069,20,"Architecture, Design, Planning","Architecture, Design, Planning",347,2007,348,2011
60520,875942,20,"Architecture, Design, Planning","Architecture, Design, Planning",150,2017,251,2018


In [5]:
# only keep the earliest survey year for each person
df_earliest = df.sort_values(by=['PersonId', 'survey_year']).drop_duplicates(subset=['PersonId', 'InstitutionId'], keep='first')
df_earliest

,PersonId,id,phd_area,prof_area,InstitutionId,DegreeYear,DegreeInstitutionID,survey_year
33960,85,15,Sociology,Sociology,300,2008,261,2011
17642,159,5,"Language, Literature, Culture","Language, Literature, Culture",26,2005,146,2011
26727,256,3,Health,Health,283,2006,343,2011
45131,276,4,Biological Sciences,Biological Sciences,148,2005,148,2011
18057,297,5,"Language, Literature, Culture","Language, Literature, Culture",274,2006,156,2011
...,...,...,...,...,...,...,...,...
44995,1489340,4,Biological Sciences,Chemical Sciences,299,2011,267,2020
44996,1500007,4,Biological Sciences,Medical Sciences,353,2011,163,2018
26518,1500295,3,Health,Health,244,2010,244,2020
41361,1500298,9,Medical Sciences,Health,244,2015,244,2020


In [6]:
# start ranking by discipline
areas = df_earliest['prof_area'].unique()
dfs = []

for area in areas:
    if not isinstance(area, str):
        continue
    print(area)
    df = filter_hiring(df_earliest.query("prof_area == @area"), multigraph=False)
    ranks = recover_ranks(df, recovered_rank_column='Rank').sort_values(by='Rank', ascending=True)
    ranks['Percentile'] = np.linspace(0, 100, len(ranks))
    ranks['area'] = area
    dfs.append(ranks)
    
df_rank = pd.concat(dfs)
df_rank

Sociology
Language, Literature, Culture
Health
Biological Sciences
Computational Sciences
Theology and Religion
History
Chemical Sciences
Political Science
Arts
Mathematical Sciences
Economics
Engineering
Business
Anthropology
Earth Sciences
Medical Sciences
Journalism, Media, Communication
Education
Psychological Sciences
Agriculture
Philosophy
Architecture, Design, Planning
Physical Sciences


,Rank,InstitutionId,Percentile,area
1,0.000000,7,0.000000,Sociology
101,0.250711,217,0.440529,Sociology
226,0.547605,539130,0.881057,Sociology
216,0.713267,395,1.321586,Sociology
214,0.714773,388,1.762115,Sociology
...,...,...,...,...
27,10.501938,49,98.181818,Physical Sciences
98,10.963708,198,98.636364,Physical Sciences
127,11.083249,251,99.090909,Physical Sciences
85,11.207247,167,99.545455,Physical Sciences


In [7]:
# save
df_rank.to_csv(r'institution_rank_by_area.csv', index=False)